# Neurite Modules

## Introduction
In this noteboook, we will demonstrate key tools found within the in `modules.py` of the `neurite`
project. These modules -- such as `Norm`, `Activation` and `ConvBlock` -- provide flexible ways to
construct n-dimensional neural networks. This notebook includes explanations, code examples, and
demonstrations to illustrate the functionalities and applications.

## Modules overview
* `Norm`: Dynamically constructs normalization n-dimensional layers. 
* `Activation`: Dynamically constructs activation functions.
* `ConvBlock`: Combines convolution, normalization, and activation into an n-dimensional sequential
block.

# Setting up
We set up our environment by telling the neurite backend that we want to use PyTorch. We will also
import torch and neurite.

In [2]:
# Standard imports
import os
import sys

# Setting up enviornment for neurite
sys.path.append('../../../')
os.environ['NEURITE_BACKEND'] = 'pytorch'

import torch
from torch import nn
import neurite as ne

In [3]:
# Let's make some dummy tensors

# 2 dimensional tensor
tensor_2d = torch.randn(1, 16, 32, 32)  # (B, C, D, H)

# 3 dimensional tensor
tensor_3d = torch.randn(1, 16, 32, 32, 32)  # (B, C, D, H, W)

# 1 Using Norm
We can easily create different normalization layers using the Norm module. It allows us to define normalization layers in different ways:
* With text: e.g. `{'instance', 'batch', 'layer'}`
* Or by passing compatable objects: e.g. `{nn.InstanceNorm3d, GroupNorm, ...}`)

## 1.1 Defining a normalization layer with text input

### 1.1.1 Batch normalization

In [4]:
norm_layer = ne.torch.modules.Norm(
    norm_type='batch',
    ndim=3,
    num_features=16
)

norm_layer(tensor_3d)
print(norm_layer)

Norm(
  (norm): BatchNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)


### 1.1.2 Instance normalization

In [4]:
norm_layer = ne.torch.modules.Norm(
    norm_type='instance',
    ndim=2,
    num_features=16
)

norm_layer(tensor_2d)
print(norm_layer)

Norm(
  (norm): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
)


### 1.1.3 Group normalization

In [5]:
norm_layer = ne.torch.modules.Norm(
    norm_type='group',
    ndim=3,
    num_features=16,
    num_groups=4   # Need to specify the number of groups
)

norm_layer(tensor_3d)
print(norm_layer)

Norm(
  (norm): GroupNorm(4, 16, eps=1e-05, affine=True)
)


## 1.2 Defining a normalization layer with object input

### 1.2.1 Instance normalization (non-instantiated)
When passing a non-instantiated normalization layer into the normalization wrapper, you'll need to provide the arguments to the wrapper.

In [6]:
norm_layer = ne.torch.modules.Norm(
    norm_type=nn.InstanceNorm3d,
    num_features=16
)

norm_layer(tensor_3d)
print(norm_layer)

Norm(
  (norm): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
)


### 1.2.2 Instance normalization (instantiated)
Let's pass an instantiated instance normalization into the `norm_type` argument. You can also pass custom normalization classes this way as well!

In [7]:
norm_layer = ne.torch.modules.Norm(
    norm_type=nn.InstanceNorm2d(
        num_features=16,
    )
)

norm_layer(tensor_2d)
print(norm_layer)

Norm(
  (norm): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
)


### 1.2.3 Group normalization (instantiated)

In [8]:
norm_layer = ne.torch.modules.Norm(
    norm_type=nn.GroupNorm(
        num_channels=16,
        num_groups=4
    )
)

norm_layer(tensor_3d)
print(norm_layer)

Norm(
  (norm): GroupNorm(4, 16, eps=1e-05, affine=True)
)


# 2 Using the `Activation` module

In [9]:
activation_layer = ne.torch.modules.Activation(
    activation_type='relu'
)

activation_layer(tensor_3d)
print(activation_layer)

Activation(
  (activation): ReLU(inplace=True)
)


In [10]:
activation_layer = ne.torch.modules.Activation(
    activation_type=None
)

activation_layer(tensor_3d)
print(activation_layer)

Activation()


In [11]:
activation_layer = ne.torch.modules.Activation(
    activation_type=nn.ELU
)

activation_layer(tensor_3d)
print(activation_layer)

Activation(
  (activation): ELU(alpha=1.0)
)


In [12]:
activation_layer = ne.torch.modules.Activation(
    activation_type=nn.LeakyReLU()
)

activation_layer(tensor_3d)
print(activation_layer)

Activation(
  (activation): LeakyReLU(negative_slope=0.01)
)


# 3 Convolutional layers

In [13]:
conv = ne.torch.modules.Conv(
    ndim=3,
    in_channels=16,
    out_channels=32
)

output_tensor = conv(tensor_3d)
print(conv)

Conv(
  (conv): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
)


# 4 Convolutional block

When defining a convolutional block, you have a lot of flexibility. For example, `norm` may be specified in many different ways.

## String based input
You can enter a string such as: {'instance', 'batch', 'group', 'layer'}
   * e.g. `norm='instance'`
   * e.g. `norm='batch'`

## Object based input
A (possibly custom) object that performs normalization, either instantiated or not.
   1. **not instantiated**: `norm=nn.InstanceNorm3d`. This will pass `nn.InstanceNorm3d` to the convlutional block and instantiate it with the information provided in the class constructor (such as `in_channels`).
   2. **instantiated**: `norm=nn.GroupNorm(num_groups=4)` will pass this instantiated class as-is to the ConvBlock constructor.

### String input

In [15]:
conv_block = ne.torch.modules.ConvBlock(
    ndim=3,
    in_channels=16,
    out_channels=32,
    norm=nn.InstanceNorm3d,
    activation=nn.ReLU(),
    order='ncaca',
)

# Visualize the convolutional block
output_tensor = conv_block(tensor_3d)
print(conv_block)

ConvBlock(
  (0): Norm(
    (norm): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
  )
  (1): Conv(
    (conv): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (2): Activation(
    (activation): ReLU()
  )
  (3): Conv(
    (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (4): Activation(
    (activation): ReLU()
  )
)


### Not instantiated
In this example, I will also show you what happens if you omit the activation layer. You will still have an activation, but it will be the predefined LeakyReLU.

In [26]:
conv_block = ne.torch.modules.ConvBlock(
    ndim=3,
    in_channels=16,
    out_channels=32,
    norm=nn.InstanceNorm3d,
    # activation=nn.ReLU(),
    order='can'
)

# Visualize the convolutional block
conv_block

ConvBlock(
  (0): Conv(
    (conv): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (1): Norm(
    (norm): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
  )
)

### Instantiated

In [7]:
conv_block = ne.torch.modules.ConvBlock(
    ndim=3,
    in_channels=16,
    out_channels=32,
    norm=nn.InstanceNorm3d(16),
    activation=nn.ReLU(),
    order='nca'
)

# Visualize the convolutional block
conv_block

ConvBlock(
  (0): Norm(
    (norm): InstanceNorm3d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
  )
  (1): Conv(
    (conv): Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  )
  (2): Activation(
    (activation): ReLU()
  )
)

In [4]:
pool = ne.torch.modules.Pool(3, 'max')